# R01 · Model-view redundancy (Table 6, Section 4.1, Fig. 4 data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-account>/nids-partition-distortion-ae-dae/blob/main/notebooks/reproduction/R01_Model_View_Redundancy.ipynb)

Recomputes Table 6 and the Section 4.1 quantities from the PRE-00 fingerprint audits, the duplicate-group manifest, the v3.4 multiplicity gate and the mixed-label group composition, and checks each value against the manuscript.

**How to run:** Runtime → Run all. The first run in a fresh Colab runtime installs the pinned library versions and restarts the runtime once; then choose Run all again.

<details><summary>Source and conversion notes</summary>

Reproduce the model-view redundancy results (manuscript Section 4.1, Table 6, Fig. 4 data).

Inputs (all inside the repository):
  artifacts/<dataset>/fingerprints/fingerprint_audit.json            PRE-00 audit counts
  artifacts/unsw_nb15/fingerprints/duplicate_group_manifest.csv      same-label groups (UNSW-NB15)
  artifacts/*/fingerprints/derived_from_archives/mixed_label_group_composition.csv
  artifacts/unsw_nb15/partition/final_source_multiplicity_balanced/{multiplicity_gate.csv,partition_integrity_report.json}
  artifacts/cicids2017/partition/attack_allocation_manifest.json
Outputs:
  results/tables/Table_3_1_model_view_redundancy.csv     (manuscript Table 6)
  results/figure_source_data/Figure_3_1_data.csv         (manuscript Fig. 4)
  results/claims_check/reproduce_fingerprints.csv

What this script cannot do: recompute the fingerprints themselves (needs the raw master CSVs and the
PRE-00 notebook) or the full-record duplicate count 480,633 (analysis record only).

Mechanical changes made when copying the .py files into this notebook:
`from src... import ...` lines are removed (the library code is defined in the cells above);
`sys.path.insert(...)` and `from __future__ import annotations` are removed;
`Path(__file__)...` repository roots are replaced by `REPO_ROOT` from the setup cell;
the command-line interface of `src/utils/hashing.py` is omitted;
command-line arguments of `reproduce_final_statistics.py` are replaced by the parameter cell;
`sys.exit(L.finish())` becomes `assert L.finish() == 0`.
</details>

In [ ]:
# ===== Setup: run this cell first =====
# REPO_SOURCE = "drive": the repository is in your Google Drive, either as the unzipped folder
#   `nids-partition-distortion-ae-dae` or as the zip file `nids-partition-distortion-ae-dae.zip`, anywhere
#   in My Drive (up to four folders deep). The cell finds it; set DRIVE_REPO only to point at one copy.
# REPO_SOURCE = "github": after the repository is on GitHub (fill in GITHUB_URL).
REPO_SOURCE = "drive"                                                    # "drive" or "github"
DRIVE_REPO = "/content/drive/MyDrive/nids-partition-distortion-ae-dae"   # optional hint
GITHUB_URL = "https://github.com/<your-account>/nids-partition-distortion-ae-dae.git"

import os, subprocess, sys, zipfile
from importlib.metadata import version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "nids-partition-distortion-ae-dae"


def _is_repo(p: Path) -> bool:
    return (p / "artifacts" / "ARTIFACT_PROVENANCE.csv").is_file()


def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if _is_repo(p):
            return p
    raise FileNotFoundError(f"repository folder not found above {start}")


def _locate_in_drive(hint: Path, drive_root: Path) -> Path:
    # 1. the hint itself, a folder above it, or a nested copy below it (unzipping often creates name/name/)
    for c in [hint, *hint.parents, hint / REPO_NAME]:
        if _is_repo(c):
            return c
    # 2. the unzipped folder anywhere in My Drive, up to four levels deep
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + ["artifacts", "ARTIFACT_PROVENANCE.csv"])
        hits = sorted(drive_root.glob(pattern))
        if hits:
            if len(hits) > 1:
                print("Several copies found; using the first (set DRIVE_REPO to choose):", *[h.parents[1] for h in hits], sep="\n  ")
            return hits[0].parents[1]
    # 3. the zip file anywhere in My Drive (Google Drive does not unzip uploads): extract it to /content
    for depth in range(0, 4):
        zips = sorted(drive_root.glob("/".join(["*"] * depth + [REPO_NAME + "*.zip"])))
        if zips:
            dest = Path("/content/repo_from_zip")
            print(f"Found {zips[0]} - extracting to {dest} (results will be written there, not to Drive)")
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            return _locate_in_drive(dest, dest)
    near = [p.name for p in drive_root.iterdir() if "nids" in p.name.lower() or "partition" in p.name.lower()] if drive_root.exists() else []
    raise FileNotFoundError(
        "Could not find the repository in Google Drive.\n"
        f"Looked for '{REPO_NAME}/artifacts/ARTIFACT_PROVENANCE.csv' (up to 4 folders deep) and for '{REPO_NAME}*.zip' in {drive_root}.\n"
        f"Items in My Drive whose names look related: {near or 'none'}.\n"
        "Fix: upload the zip file (or the unzipped folder) to My Drive, or set DRIVE_REPO to the folder that "
        "contains README.md, artifacts/ and notebooks/.")


_previous = globals().get("REPO_ROOT")                 # set by an earlier notebook in the same runtime (R00)
if _previous is not None and _is_repo(Path(_previous)):
    REPO_ROOT = Path(_previous)
elif IN_COLAB and REPO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = _locate_in_drive(Path(DRIVE_REPO), Path("/content/drive/MyDrive"))
elif IN_COLAB:
    target = Path("/content/" + REPO_NAME)
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(target)], check=True)
    REPO_ROOT = _find_repo(target)
else:                                     # local Jupyter: the notebook lives in notebooks/reproduction/
    REPO_ROOT = _find_repo(Path.cwd().resolve())
os.chdir(REPO_ROOT)

# The results were verified with these exact versions (the versions recorded by the Colab analysis).
PINNED = {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.16.3", "scikit-learn": "1.6.1",
          "matplotlib": "3.10.0", "pytest": "8.3.5"}
wrong = {k: v for k, v in PINNED.items() if version(k) != v}
if wrong and IN_COLAB:
    print("Installing the pinned versions:", wrong)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
    print("Done. The runtime will now restart; then run the notebook again from the top (Runtime > Run all).")
    os.kill(os.getpid(), 9)
elif wrong:
    print("WARNING: library versions differ from requirements.txt:", wrong)
print("Repository:", REPO_ROOT)
print("Python", sys.version.split()[0], "|", ", ".join(f"{k} {version(k)}" for k in PINNED))

## Library code
Verbatim copies of the repository modules this notebook uses (`src/`). Run these cells; there is nothing to edit.

In [ ]:
# ===== src/utils/io_utils.py =====
"""Paths and loaders shared by the reproduction scripts and tests."""

import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(REPO_ROOT)  # set by the setup cell
ARTIFACTS = REPO_ROOT / "artifacts"
RESULTS = REPO_ROOT / "results"
DATASETS = {"cicids2017": "CICIDS2017", "unsw_nb15": "UNSW-NB15"}

# Columns of every observation-level prediction archive written by NB-03/NB-07.
ARCHIVE_COLUMNS = ["row_id", "y_true", "reconstruction_error", "threshold", "y_pred", "model", "seed",
                   "config_hash", "run_hash", "attack_family", "feature_fingerprint"]
ARCHIVE_DTYPES = {"row_id": str, "y_true": "int8", "reconstruction_error": "float64", "threshold": "float64",
                  "y_pred": "int8", "model": str, "seed": "int64", "config_hash": str, "run_hash": str,
                  "attack_family": str, "feature_fingerprint": str}
# CICIDS2017 web-attack family names contain U+FFFD where the original CSVs had a cp1252 en dash
# ("Web Attack � Brute Force"); the character is kept as stored in the archives.

# Environment variable that points to the folder holding the external prediction archives.
ARCHIVE_ENV = "NIDS_PREDICTION_ARCHIVES"


def art(*parts: str) -> Path:
    """Path inside ``artifacts/``."""
    return ARTIFACTS.joinpath(*parts)


def load_json(path: str | Path) -> dict:
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def read_csv(path: str | Path, **kw) -> pd.DataFrame:
    """``pandas.read_csv`` with exact (round-trip) float parsing, so that committed tables compare exactly."""
    kw.setdefault("float_precision", "round_trip")
    return pd.read_csv(path, **kw)


def read_prediction_archive(path: str | Path) -> pd.DataFrame:
    """Read one observation-level prediction archive with the dtypes used in the study."""
    return pd.read_csv(path, usecols=ARCHIVE_COLUMNS, dtype=ARCHIVE_DTYPES)


def archive_dir(cli_value: str | None = None) -> Path | None:
    """Folder with the external prediction archives (CLI argument first, then the environment variable)."""
    value = cli_value or os.environ.get(ARCHIVE_ENV)
    if not value:
        return None
    p = Path(value).expanduser()
    return p if p.is_dir() else None


def dataset_of_row_id(row_id: str) -> str:
    """``row_id`` is ``<dataset>::<source file>::<10-digit source row>``; return the repository dataset key."""
    prefix = row_id.split("::", 1)[0]
    return {"CICIDS2017": "cicids2017", "UNSWNB15": "unsw_nb15"}[prefix]


def role_sizes(ds: str) -> dict:
    """Evaluation-role sizes recorded in the frozen result tables (TP+FP+FN+TN of any run)."""
    fin = read_csv(art(ds, "final_evaluation", "final_seed_results.csv")).iloc[0]
    dev = read_csv(art(ds, "model_selection", "confirmation_seed_results.csv")).iloc[0]
    tot = lambda r: int(r.TP + r.FP + r.FN + r.TN)
    return {"final_test": tot(fin), "development": tot(dev)}


def write_csv(df: pd.DataFrame, path: str | Path, **kw) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator="\n", **kw)
    return path

In [ ]:
# ===== src/utils/seeds.py =====
"""Seed register of Experiment 1 v2.

Every constant below is read back from a frozen artefact by ``tests/test_seed_register.py``;
the module does not set any random state for model training (training ran in the Colab notebooks).
"""
FINAL_SEEDS = (101, 202, 303, 404, 505)          # final_seed_results.csv
CONFIRMATION_SEEDS = (101, 202, 303)             # selected_models_manifest.json["confirmation_seeds"]
PARTITION_SEED = 20260914                        # partition_policy_manifest.json / integrity reports
BOOTSTRAP_SEED = 2026                            # paired_seed_bootstrap_effects.csv["bootstrap_seed"]
BOOTSTRAP_ITERATIONS = 10_000                    # paired_seed_bootstrap_effects.csv["bootstrap_iterations"]
CONFIDENCE_LEVEL = 0.95                          # paired_seed_bootstrap_effects.csv["confidence_level"]

In [ ]:
# ===== src/utils/claims.py =====
"""Record manuscript claims next to the values recomputed from the artefacts.

Each check compares a recomputed number with the value *as printed in the manuscript*. The tolerance is
half a unit in the last printed decimal place, so "0.7094" passes for any value in [0.70935, 0.70945].
Every reproduction script writes its log to ``results/claims_check/<script>.csv`` and exits with a
non-zero status if any check fails.
"""

import math
import sys
from pathlib import Path

import pandas as pd



def _parse(printed: str) -> tuple[float, float]:
    s = str(printed).replace(",", "").replace("%", "").replace("+", "").replace("−", "-").strip()
    dec = len(s.split(".")[1]) if "." in s and "e" not in s.lower() else 0
    if "e" in s.lower():
        mant = s.lower().split("e")[0]
        exp = int(s.lower().split("e")[1])
        dec_m = len(mant.split(".")[1]) if "." in mant else 0
        return float(s), 0.5 * 10 ** (exp - dec_m)
    return float(s), 0.5 * 10 ** (-dec)


class ClaimLog:
    def __init__(self, script: str):
        self.script = script
        self.rows: list[dict] = []

    def check(self, claim_id: str, location: str, description: str, reproduced, printed: str,
              source: str, scale: float = 1.0) -> bool:
        """``scale`` converts the reproduced value to the printed unit (e.g. 100 for percentages)."""
        value, tol = _parse(printed)
        rep = float(reproduced) * scale
        ok = math.isclose(rep, value, abs_tol=tol + 1e-12)
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": printed, "reproduced_value": repr(round(rep, 10)),
                          "status": "PASS" if ok else "FAIL", "source": source})
        return ok

    def assert_true(self, claim_id: str, location: str, description: str, condition: bool, source: str) -> bool:
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "true", "reproduced_value": str(bool(condition)),
                          "status": "PASS" if condition else "FAIL", "source": source})
        return bool(condition)

    def note(self, claim_id: str, location: str, description: str, value: str, source: str, status: str) -> None:
        """Record a finding that is not a pass/fail comparison (e.g. a discrepancy that the manuscript notes)."""
        self.rows.append({"claim_id": claim_id, "manuscript_location": location, "description": description,
                          "manuscript_value": "", "reproduced_value": value, "status": status, "source": source})

    def finish(self) -> int:
        df = pd.DataFrame(self.rows)
        out = RESULTS / "claims_check" / f"{self.script}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out, index=False, lineterminator="\n")
        n_fail = int((df.status == "FAIL").sum())
        n_pass = int((df.status == "PASS").sum())
        print(f"[{self.script}] {n_pass} checks passed, {n_fail} failed, {len(df) - n_pass - n_fail} notes -> {out}")
        if n_fail:
            print(df[df.status == "FAIL"].to_string(index=False), file=sys.stderr)
        return 1 if n_fail else 0

In [ ]:
# ===== src/fingerprinting/multiplicity_audit.py =====
"""Fingerprint-multiplicity bands, total-variation distance and related structural quantities.

The eight frozen bands are those of the UNSW-NB15 source-multiplicity amendment
(``source_multiplicity_amendment.json["bands"]``): 1, 2, 3-4, 5-9, 10-19, 20-49, 50-99, >=100.
"""

import numpy as np
import pandas as pd

BANDS = ["1", "2", "3-4", "5-9", "10-19", "20-49", "50-99", ">=100"]
EDGES = [0, 1, 2, 4, 9, 19, 49, 99, np.inf]   # right-closed intervals (0,1], (1,2], (2,4], ...


def band_of(group_size) -> pd.Series:
    """Map fingerprint-group sizes (multiplicities) to the frozen band labels."""
    return pd.cut(pd.Series(group_size), bins=EDGES, labels=BANDS, right=True).astype(str)


def shares(rows_by_band: pd.Series) -> pd.Series:
    """Band shares of rows, in the frozen band order."""
    r = rows_by_band.reindex(BANDS).fillna(0).astype(float)
    return r / r.sum()


def tv_distance(p: pd.Series, q: pd.Series) -> float:
    """Total-variation distance 1/2 * sum_b |p_b - q_b| between two band-share vectors."""
    return float(0.5 * (shares(p) - shares(q)).abs().sum())


def gate_rows(gate: pd.DataFrame, column: str = "actual_rows") -> pd.DataFrame:
    """Role x band row matrix from ``multiplicity_gate.csv``."""
    return gate.pivot(index="partition", columns="size_band", values=column)[BANDS]


def expected_cross_side_sharing(group_sizes: np.ndarray) -> float:
    """Expected number of rows sharing their fingerprint with >=1 row on the opposite side when every row
    is assigned independently to one of two sides with probability 1/2: sum_g m_g (1 - 2^{-(m_g - 1)})."""
    m = np.asarray(group_sizes, dtype=float)
    return float((m * (1.0 - np.power(0.5, m - 1.0))).sum())


def redundancy_rate(rows: int, groups: int) -> float:
    """(rows - groups) / rows: the share of rows that repeat a model-view vector already present."""
    return (rows - groups) / rows

In [ ]:
# ===== src/fingerprinting/duplicate_audit.py =====
"""Model-view redundancy accounting from the frozen PRE-00 fingerprint audits and group manifests.

The fingerprints themselves cannot be recomputed here (that requires the raw master CSVs and the
PRE-00 notebook); this module recomputes every *count* the manuscript derives from the frozen audit
outputs, and documents a property of the stored fingerprint strings.
"""

import hashlib

import numpy as np
import pandas as pd


def audit_counts(audit: dict) -> dict:
    """Corpus-level redundancy quantities from ``fingerprint_audit.json`` (Table 6 rows)."""
    n = audit["total_observations"]
    g = audit["number_of_fingerprint_groups"]
    nonsingleton_rows = (audit["number_of_observations_in_same_label_duplicate_groups"]
                         + audit["number_of_observations_in_mixed_label_groups"])
    return {
        "master_rows": n,
        "fingerprint_groups": g,
        "singleton_groups": audit["number_of_unique_fingerprint_groups"],
        "same_label_duplicate_groups": audit["number_of_same_label_duplicate_groups"],
        "same_label_duplicate_rows": audit["number_of_observations_in_same_label_duplicate_groups"],
        "mixed_label_groups": audit["number_of_mixed_label_fingerprint_groups"],
        "mixed_label_rows": audit["number_of_observations_in_mixed_label_groups"],
        "rows_in_nonsingleton_groups": nonsingleton_rows,
        "rows_in_nonsingleton_groups_pct": 100.0 * nonsingleton_rows / n,
        "modelview_duplicates_beyond_first": n - g,
        "modelview_duplicates_beyond_first_pct": 100.0 * (n - g) / n,
        "largest_group": audit["maximum_fingerprint_group_size"],
        "collision_groups_resolved": audit["resolved_effective_128bit_collision_groups"],
        "groups_reconcile": audit["number_of_unique_fingerprint_groups"] + audit["number_of_same_label_duplicate_groups"]
        + audit["number_of_mixed_label_fingerprint_groups"] == g,
        "rows_reconcile": audit["number_of_unique_fingerprint_groups"] + nonsingleton_rows == n,
    }


def error_floor(benign_rows: np.ndarray, attack_rows: np.ndarray) -> int:
    """Irreducible error of any deterministic detector on mixed-label groups: sum_g min(benign_g, attack_g)."""
    return int(np.minimum(np.asarray(benign_rows), np.asarray(attack_rows)).sum())


def pure_benign_reconciliation(gate: pd.DataFrame, duplicate_manifest: pd.DataFrame) -> dict:
    """Pure-benign rows and fingerprints of UNSW-NB15 from the v3.4 multiplicity gate and the same-label
    duplicate manifest. Singletons are counted once per row; each benign duplicate group once."""
    rows = int(gate.actual_rows.sum())
    singletons = int(gate.loc[gate.size_band == "1", "actual_rows"].sum())
    benign_dup = duplicate_manifest[duplicate_manifest.binary_label_max == 0]
    dup_rows = int(benign_dup.fingerprint_group_size.sum())
    return {"pure_benign_rows": rows, "pure_benign_singletons": singletons,
            "pure_benign_duplicate_groups": len(benign_dup), "pure_benign_duplicate_rows": dup_rows,
            "pure_benign_fingerprints": singletons + len(benign_dup),
            "reconciles": rows - singletons == dup_rows}


def fingerprint_halves_identical(fingerprints: pd.Series) -> float:
    """Share of ``h128:`` fingerprints whose two 64-bit halves are identical.

    pandas ``hash_pandas_object`` applies ``hash_key`` only to object (string) columns, so for an
    all-numeric model view (CICIDS2017) two "independently keyed" 64-bit hashes coincide and the
    candidate key carries 64 effective bits. Group membership was verified by byte equality in
    PRE-00, so the group counts are unaffected; the "128 effective bits" description is not.
    """
    h = fingerprints.astype(str).str.removeprefix("h128:")
    return float((h.str[:16] == h.str[16:32]).mean())


def tie_break_key(partition_seed: int, fingerprint: str) -> str:
    """Deterministic tie-breaking key of the CICIDS2017 partition policy: SHA-256(str(seed) + '|' + fingerprint)."""
    return hashlib.sha256(f"{partition_seed}|{fingerprint}".encode("utf-8")).hexdigest()

## Reproduction: `scripts/reproduce_fingerprints.py`

In [ ]:
import sys
from pathlib import Path


import pandas as pd  # noqa: E402


L = ClaimLog("reproduce_fingerprints")
FINAL = ("unsw_nb15", "partition", "final_source_multiplicity_balanced")

### UNSW-NB15

In [ ]:
au = load_json(art("unsw_nb15", "fingerprints", "fingerprint_audit.json"))
cu = audit_counts(au)
gate = read_csv(art(*FINAL, "multiplicity_gate.csv"))
integ = load_json(art(*FINAL, "partition_integrity_report.json"))
dup = read_csv(art("unsw_nb15", "fingerprints", "duplicate_group_manifest.csv"))
mix_u = read_csv(art("unsw_nb15", "fingerprints", "derived_from_archives", "mixed_label_group_composition.csv"))
pb = pure_benign_reconciliation(gate, dup)
u_benign = pb["pure_benign_rows"] + int(mix_u.benign_rows.sum())
u_attack = cu["master_rows"] - u_benign
u_floor = error_floor(mix_u.benign_rows, mix_u.attack_rows)
all_group_sizes = pd.concat([dup.fingerprint_group_size, mix_u.rows]).to_numpy()
u_cross = expected_cross_side_sharing(all_group_sizes)

src_a = "artifacts/unsw_nb15/fingerprints/fingerprint_audit.json"
for k, v, p in [("master_rows", cu["master_rows"], "2,540,047"), ("fingerprint_groups", cu["fingerprint_groups"], "2,016,862"),
                ("singleton_groups", cu["singleton_groups"], "1,940,641"),
                ("same_label_duplicate_groups", cu["same_label_duplicate_groups"], "75,768"),
                ("same_label_duplicate_rows", cu["same_label_duplicate_rows"], "597,527"),
                ("mixed_label_groups", cu["mixed_label_groups"], "453"), ("mixed_label_rows", cu["mixed_label_rows"], "1,879"),
                ("rows_in_nonsingleton_groups", cu["rows_in_nonsingleton_groups"], "599,406"),
                ("modelview_duplicates_beyond_first", cu["modelview_duplicates_beyond_first"], "523,185"),
                ("largest_group", cu["largest_group"], "760")]:
    L.check(f"T6.unsw.{k}", "Table 6; Sec. 4.1", k.replace("_", " "), v, p, src_a)
L.check("T6.unsw.nonsingleton_pct", "Table 6; Sec. 4.1", "share of rows in non-singleton groups", cu["rows_in_nonsingleton_groups_pct"], "23.60", src_a)
L.check("T6.unsw.dup_pct", "Sec. 4.1", "model-view duplicates beyond first occurrence (%)", cu["modelview_duplicates_beyond_first_pct"], "20.5975", src_a)
L.check("abstract.unsw.dup_pct", "Results opening; Table 6", "UNSW-NB15 redundancy (%)", cu["modelview_duplicates_beyond_first_pct"], "20.60", src_a)
L.assert_true("T6.unsw.reconciles", "Table 6", "singletons + duplicate groups + mixed groups = groups; rows reconcile",
              cu["groups_reconcile"] and cu["rows_reconcile"], src_a)
L.check("T6.unsw.benign_rows", "Table 6", "benign rows = pure-benign rows + benign members of mixed groups", u_benign, "2,218,764",
        "multiplicity_gate.csv + mixed_label_group_composition.csv")
L.check("T6.unsw.attack_rows", "Table 6", "attack rows = master rows - benign rows", u_attack, "321,283", "as above")
L.check("T6.unsw.pure_benign_rows", "Table 6; abstract", "pure-benign rows (gate)", pb["pure_benign_rows"], "2,217,778", "multiplicity_gate.csv")
L.check("T6.unsw.pure_benign_rows_integrity", "Table 6", "pure-benign rows (integrity report)", integ["pure_benign_rows"], "2,217,778",
        "partition_integrity_report.json")
L.check("T6.unsw.pure_benign_fps", "Table 6; abstract", "pure-benign fingerprints = singletons + benign duplicate groups",
        pb["pure_benign_fingerprints"], "1,938,413", "multiplicity_gate.csv + duplicate_group_manifest.csv")
L.check("T6.unsw.pure_benign_fps_integrity", "Table 6", "pure-benign fingerprints (integrity report)",
        integ["pure_benign_fingerprints"], "1,938,413", "partition_integrity_report.json")
L.assert_true("S1.fig4.reconciliation", "Supplementary S1", "pure-benign non-singleton rows equal benign duplicate-group rows (335,533)",
              pb["reconciles"] and pb["pure_benign_duplicate_rows"] == 335533 and pb["pure_benign_duplicate_groups"] == 56168,
              "multiplicity_gate.csv + duplicate_group_manifest.csv")
L.check("S4.1.repeat_rows", "Sec. 4.1", "pure-benign rows repeating a vector already present",
        pb["pure_benign_rows"] - pb["pure_benign_fingerprints"], "279,365", "as above")
L.check("S4.1.repeat_pct", "Sec. 4.1", "share of pure-benign rows repeating a vector (%)",
        100 * (pb["pure_benign_rows"] - pb["pure_benign_fingerprints"]) / pb["pure_benign_rows"], "12.60", "as above")
L.check("S4.1.benign_in_mixed", "Sec. 4.1", "benign rows in mixed-label groups", int(mix_u.benign_rows.sum()), "986",
        "mixed_label_group_composition.csv")
L.check("T6.unsw.mixed_groups_archive", "Sec. 4.1", "mixed-label groups present in the development archives", len(mix_u), "453",
        "mixed_label_group_composition.csv")
L.check("T6.unsw.error_floor", "Table 6; Sec. 4.1", "irreducible error floor sum_g min(benign_g, attack_g)", u_floor, "706",
        "mixed_label_group_composition.csv")
L.check("T6.unsw.error_floor_pct", "Table 6", "error floor (% of corpus)", 100 * u_floor / cu["master_rows"], "0.0278", "as above")
L.check("S4.1.cross_side", "Sec. 4.1", "expected rows sharing a fingerprint across two random halves", u_cross, "549,885",
        "duplicate_group_manifest.csv + mixed_label_group_composition.csv")
L.check("S4.1.cross_side_pct", "Sec. 4.1", "expected cross-side sharing (% of corpus)", 100 * u_cross / cu["master_rows"], "21.6486", "as above")
L.note("S4.1.full_record_duplicates", "Sec. 4.1; Table 6", "full-record duplicates 480,633 and 568,843 and the 42,552 difference",
       "not recomputable: needs the raw master CSV", "analysis record (PRE-00 canonicalisation)", "NOT_REPRODUCIBLE_HERE")

### CICIDS2017

In [ ]:
ac = load_json(art("cicids2017", "fingerprints", "fingerprint_audit.json"))
cc = audit_counts(ac)
am = load_json(art("cicids2017", "partition", "attack_allocation_manifest.json"))
mg = pd.DataFrame(am["mixed_label_group_assignments"])
mix_c = read_csv(art("cicids2017", "fingerprints", "derived_from_archives", "mixed_label_group_composition.csv"))
c_floor = error_floor(mg.benign_rows, mg.attack_rows)
c_pure_benign = am["B_total"] - int(mg.benign_rows.sum())
src_c = "artifacts/cicids2017/fingerprints/fingerprint_audit.json"
for k, v, p in [("master_rows", cc["master_rows"], "2,830,743"), ("fingerprint_groups", cc["fingerprint_groups"], "2,521,664"),
                ("singleton_groups", cc["singleton_groups"], "2,426,185"),
                ("same_label_duplicate_groups", cc["same_label_duplicate_groups"], "94,781"),
                ("same_label_duplicate_rows", cc["same_label_duplicate_rows"], "397,538"),
                ("mixed_label_groups", cc["mixed_label_groups"], "698"), ("mixed_label_rows", cc["mixed_label_rows"], "7,020"),
                ("rows_in_nonsingleton_groups", cc["rows_in_nonsingleton_groups"], "404,558"),
                ("modelview_duplicates_beyond_first", cc["modelview_duplicates_beyond_first"], "309,079"),
                ("largest_group", cc["largest_group"], "9,329")]:
    L.check(f"T6.cic.{k}", "Table 6; Sec. 4.6", k.replace("_", " "), v, p, src_c)
L.check("T6.cic.nonsingleton_pct", "Table 6", "share of rows in non-singleton groups", cc["rows_in_nonsingleton_groups_pct"], "14.29", src_c)
L.check("T6.cic.dup_pct", "Table 6; Sec. 4.6", "model-view duplicates beyond first occurrence (%)", cc["modelview_duplicates_beyond_first_pct"], "10.92", src_c)
L.assert_true("T6.cic.reconciles", "Table 6", "group and row accounting reconcile", cc["groups_reconcile"] and cc["rows_reconcile"], src_c)
L.check("T6.cic.benign_rows", "Table 6", "benign rows", am["B_total"], "2,273,097", "attack_allocation_manifest.json")
L.check("T6.cic.attack_rows", "Table 6", "attack rows", am["A_total"], "557,646", "attack_allocation_manifest.json")
L.check("T6.cic.pure_benign_rows", "Table 6 note 2", "benign rows minus benign members of mixed groups", c_pure_benign, "2,272,361",
        "attack_allocation_manifest.json")
L.check("T6.cic.benign_in_mixed", "Table 6 note 2", "benign members of mixed groups", int(mg.benign_rows.sum()), "736", "attack_allocation_manifest.json")
L.check("T6.cic.error_floor", "Table 6; Sec. 4.6", "irreducible error floor", c_floor, "722", "attack_allocation_manifest.json")
L.check("T6.cic.error_floor_pct", "Table 6", "error floor (% of corpus)", 100 * c_floor / cc["master_rows"], "0.0255", "as above")
L.check("S4.8.cic_mixed_attacks", "Sec. 4.7", "attack observations in mixed-label groups", int(mg.attack_rows.sum()), "6,284", "attack_allocation_manifest.json")
same = mix_c.set_index("feature_fingerprint")[["benign_rows", "attack_rows"]].sort_index().equals(
    mg.set_index("feature_fingerprint")[["benign_rows", "attack_rows"]].sort_index())
L.assert_true("X.cic.mixed_manifest_vs_archive", "Supplementary S1", "mixed-group label counts in the archives equal the allocation manifest",
              same, "mixed_label_group_composition.csv vs attack_allocation_manifest.json")
L.assert_true("X.cic.mixed_all_development", "Sec. 4.6", "all 698 mixed-label groups were assigned to development",
              set(mg.partition) == {"development"} and len(mg) == 698, "attack_allocation_manifest.json")
L.assert_true("X.cic.tie_keys", "Methods (tie-breaking)", "stable_tie_key = SHA-256(str(PARTITION_SEED)|fingerprint) for all 698 groups",
              all(tie_break_key(PARTITION_SEED, f) == k for f, k in zip(mg.feature_fingerprint, mg.stable_tie_key)),
              "attack_allocation_manifest.json + partition_policy_manifest.json")

### fingerprint string property (see LIMITATIONS)

In [ ]:
half_c = fingerprint_halves_identical(read_csv(art("cicids2017", "partition", "partition_membership_first100k.csv"),
                                               usecols=["feature_fingerprint"]).feature_fingerprint)
half_u = fingerprint_halves_identical(read_csv(art(*FINAL, "partition_membership_first100k.csv"),
                                               usecols=["feature_fingerprint"]).feature_fingerprint)
L.note("X.fingerprint_bits", "Supplementary S1 (fingerprint)", "share of fingerprints whose two 64-bit halves are identical",
       f"CICIDS2017 {half_c:.4f}; UNSW-NB15 {half_u:.4f}", "partition_membership_first100k.csv (both corpora)",
       "CORRECTION_NEEDED" if half_c == 1.0 else "INFO")

### outputs

In [ ]:
rows = [
    ("Master rows", cu["master_rows"], cc["master_rows"]), ("Benign rows", u_benign, am["B_total"]),
    ("Attack rows", u_attack, am["A_total"]), ("Fingerprint groups", cu["fingerprint_groups"], cc["fingerprint_groups"]),
    ("Singleton groups", cu["singleton_groups"], cc["singleton_groups"]),
    ("Same-label duplicate groups", cu["same_label_duplicate_groups"], cc["same_label_duplicate_groups"]),
    ("Rows in same-label duplicate groups", cu["same_label_duplicate_rows"], cc["same_label_duplicate_rows"]),
    ("Mixed-label groups", cu["mixed_label_groups"], cc["mixed_label_groups"]),
    ("Rows in mixed-label groups", cu["mixed_label_rows"], cc["mixed_label_rows"]),
    ("Rows in non-singleton groups", cu["rows_in_nonsingleton_groups"], cc["rows_in_nonsingleton_groups"]),
    ("Rows in non-singleton groups (%)", round(cu["rows_in_nonsingleton_groups_pct"], 2), round(cc["rows_in_nonsingleton_groups_pct"], 2)),
    ("Model-view duplicate rows beyond first occurrence", cu["modelview_duplicates_beyond_first"], cc["modelview_duplicates_beyond_first"]),
    ("Model-view duplicate rows beyond first occurrence (%)", round(cu["modelview_duplicates_beyond_first_pct"], 2),
     round(cc["modelview_duplicates_beyond_first_pct"], 2)),
    ("Exact full-record duplicate rows beyond first occurrence", "480633 (analysis record; not recomputable here)", "not available"),
    ("Pure-benign rows", pb["pure_benign_rows"], c_pure_benign),
    ("Pure-benign fingerprint groups", pb["pure_benign_fingerprints"], "not available"),
    ("Irreducible error floor (rows)", u_floor, c_floor),
    ("Irreducible error floor (%)", round(100 * u_floor / cu["master_rows"], 4), round(100 * c_floor / cc["master_rows"], 4)),
    ("Largest group (rows)", cu["largest_group"], cc["largest_group"]),
]
write_csv(pd.DataFrame(rows, columns=["quantity", "UNSW-NB15", "CICIDS2017"]),
          RESULTS / "tables" / "Table_3_1_model_view_redundancy.csv")
fig = pd.DataFrame({"panel": ["UNSW-NB15 all", "UNSW-NB15 pure benign", "CICIDS2017 all"],
                    "rows": [cu["master_rows"], pb["pure_benign_rows"], cc["master_rows"]],
                    "fingerprint_groups": [cu["fingerprint_groups"], pb["pure_benign_fingerprints"], cc["fingerprint_groups"]]})
fig["redundancy_pct"] = 100 * (fig.rows - fig.fingerprint_groups) / fig.rows
write_csv(fig, RESULTS / "figure_source_data" / "Figure_3_1_data.csv")
assert L.finish() == 0, "some checks FAILED - see the claim log below"

## Results

In [ ]:
# ===== Results of the checks =====
import pandas as pd
from IPython.display import display
log = pd.read_csv(REPO_ROOT / "results" / "claims_check" / "reproduce_fingerprints.csv")
print(log.status.value_counts().to_string())
display(log[log.status != "PASS"])          # notes: values that cannot be recomputed here, or corrections
display(log.head(15))                        # first passing checks (all rows are in the CSV)

In [ ]:
# ===== Tables written by this notebook =====
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 30)
print("results/tables/Table_3_1_model_view_redundancy.csv"); display(pd.read_csv(REPO_ROOT / "results/tables/Table_3_1_model_view_redundancy.csv"))